# 04 — Explainability with SHAP

Implements **plan §11**. Loads the persisted champion, builds a `FraudExplainer`,
and produces both global (summary, bar) and local (waterfall) explanations.
Includes a SHAP-vs-XGBoost gain comparison per §11.1.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from src.data import DEFAULT_DATA_PATH, load_raw, split_stratified
from src.explain import FraudExplainer

SEED = 42
shap.initjs()

In [ ]:
df = load_raw(ROOT / DEFAULT_DATA_PATH)
splits = split_stratified(df, random_state=SEED)
champion = joblib.load(ROOT / 'models' / 'champion.pkl')
explainer = FraudExplainer(champion)
print('Feature space size:', len(explainer.feature_names))
print('Base value (log-odds):', explainer.expected_value())

## 1. Global explanations (§11.1)

In [ ]:
# Beeswarm summary (sample of 10k test rows per §11.4 — full set is overkill).
explainer.summary_plot(splits.X_test, sample=10_000, seed=SEED)

In [ ]:
# Mean absolute SHAP bar plot — most defensible feature importance for tree
# ensembles per §11.1.
explainer.bar_plot(splits.X_test, sample=10_000, seed=SEED)

In [ ]:
# SHAP-vs-XGBoost-gain importance comparison (§11.1).
model = champion.named_steps['model']
fe = champion.named_steps['features']
X_sample = splits.X_test.sample(n=10_000, random_state=SEED)
X_sample_fe = fe.transform(X_sample)
sv = explainer.shap_values(X_sample)
shap_imp = np.abs(sv).mean(axis=0)

feature_names = explainer.feature_names
booster = model.get_booster()
gain = booster.get_score(importance_type='gain')
# Booster names are 'f0', 'f1', ... mapped by column index.
gain_arr = np.zeros(len(feature_names))
for k, v in gain.items():
    idx = int(k.lstrip('f'))
    if idx < len(gain_arr):
        gain_arr[idx] = v

imp_df = (pd.DataFrame({
    'feature': feature_names,
    'shap_mean_abs': shap_imp,
    'xgb_gain': gain_arr,
})
    .sort_values('shap_mean_abs', ascending=False)
    .reset_index(drop=True))
imp_df['shap_rank'] = imp_df['shap_mean_abs'].rank(ascending=False).astype(int)
imp_df['gain_rank'] = imp_df['xgb_gain'].rank(ascending=False).astype(int)
imp_df['rank_diff'] = (imp_df['shap_rank'] - imp_df['gain_rank']).astype(int)
imp_df.head(15)

Per §11.1: disagreement between SHAP rank and gain rank is informative. SHAP
is the more reliable signal because gain double-counts high-cardinality features.

## 2. Local explanations (§11.2)

Pick 2 TPs, 2 TNs, 2 FPs, 2 FNs from the test set and explain each. The FP
and FN explanations are the most interesting per the plan.

In [ ]:
import json
metadata = json.loads((ROOT / 'models' / 'metadata.json').read_text())
threshold = metadata['deployment']['chosen_threshold']
proba_test = champion.predict_proba(splits.X_test)[:, 1]
pred = (proba_test >= threshold).astype(int)
y_true = splits.y_test.values

tp_idx = np.where((pred == 1) & (y_true == 1))[0][:2]
tn_idx = np.where((pred == 0) & (y_true == 0))[0][:2]
fp_idx = np.where((pred == 1) & (y_true == 0))[0][:2]
fn_idx = np.where((pred == 0) & (y_true == 1))[0][:2]
samples = {
    'TP': tp_idx,
    'TN': tn_idx,
    'FP': fp_idx,
    'FN': fn_idx,
}
for label, idxs in samples.items():
    print(f'{label}: indices={list(idxs)} probs={[round(float(proba_test[i]), 4) for i in idxs]}')

In [ ]:
# Waterfall plots for the eight selected rows.
for label, idxs in samples.items():
    for i in idxs:
        print(f'--- {label} row #{i}  prob={proba_test[i]:.4f} ---')
        explainer.waterfall_for_row(splits.X_test.reset_index(drop=True), row_idx=int(i))
        plt.show()

## 3. SHAP dependence plots (§11.3)

For the top-3 features, plot SHAP value vs. feature value. Reveals nonlinear
relationships and interaction effects.

In [ ]:
top_features = imp_df['feature'].head(3).tolist()
for name in top_features:
    fig, ax = plt.subplots(figsize=(7, 4))
    col_idx = explainer.feature_names.index(name)
    ax.scatter(X_sample_fe[name].values, sv[:, col_idx], s=4, alpha=0.4)
    ax.set_xlabel(name)
    ax.set_ylabel(f'SHAP({name})')
    ax.set_title(f'SHAP dependence — {name}')
    ax.axhline(0, color='grey', lw=0.5)
    plt.tight_layout()
    plt.show()

## 4. API smoke test

Confirms the SHAP top-3 helper that the API uses returns sensible output.

In [ ]:
tops = explainer.top_contributors(splits.X_test.reset_index(drop=True).iloc[:5], k=3)
for i, contribs in enumerate(tops):
    print(f'Row {i}: prob={proba_test[i]:.4f}')
    for c in contribs:
        print(f'  - {c.name:>22}  value={c.value:+.4f}  shap={c.shap_value:+.4f}')
    print()